In [1]:
import json
import requests
import numpy as np
import pandas as pd

In [2]:
with open('../key_ring.json') as fi:
    credentials = json.load(fi)

In [3]:
api_key = credentials['Census_ACS']

In [4]:
years = np.arange(2009, 2025)  ## Creating an array to make sure I can get data from 2009 - 2024

## DP05  
I'll use the ACS Demographic & Housing Estimates to get total population estimates for each state.  

* DP05_0001E: total population estimate
* DP05_0001M: margin of error for total population estimate


In [5]:
DP05_params = {
    'get': 'NAME,GEO_ID,DP05_0001E,DP05_0001M', ## field names, separated by comma
    'for': 'state:*',
    'key': api_key
}

In [6]:
## Create empty dataframe to fill
DP05_df = pd.DataFrame()

## loop through the years in range
for year in years:
    endpoint = 'https://api.census.gov/data/' + str(year) +'/acs/acs5/profile?'  ## create endpoint URL incorporating year
    response = requests.get(endpoint, params = DP05_params).json()  ## get request from API
    df = pd.DataFrame(response) ## convert to dataframe
    df.columns = df.iloc[0] ## promote first row to headers
    df = df[1:] ## drop first row
    df['year'] = year ## add year

    DP05_df = pd.concat([DP05_df, df]) ## add current year dataframe to the overall dataframe

DP05_df

,NAME,GEO_ID,DP05_0001E,DP05_0001M,state,year
1,Alabama,0400000US01,4633360,0,01,2009
2,Alaska,0400000US02,683142,0,02,2009
3,Arizona,0400000US04,6324865,0,04,2009
4,Arkansas,0400000US05,2838143,0,05,2009
5,California,0400000US06,36308527,0,06,2009
...,...,...,...,...,...,...
48,Washington,0400000US53,7816116,-555555555,53,2024
49,West Virginia,0400000US54,1778373,-555555555,54,2024
50,Wisconsin,0400000US55,5914872,-555555555,55,2024
51,Wyoming,0400000US56,582397,-555555555,56,2024


In [7]:
DP05_df = DP05_df.rename(columns = {'DP05_0001E': 'pop_est', 'DP05_0001M': 'pop_est_me'})

In [8]:
DP05_df

,NAME,GEO_ID,pop_est,pop_est_me,state,year
1,Alabama,0400000US01,4633360,0,01,2009
2,Alaska,0400000US02,683142,0,02,2009
3,Arizona,0400000US04,6324865,0,04,2009
4,Arkansas,0400000US05,2838143,0,05,2009
5,California,0400000US06,36308527,0,06,2009
...,...,...,...,...,...,...
48,Washington,0400000US53,7816116,-555555555,53,2024
49,West Virginia,0400000US54,1778373,-555555555,54,2024
50,Wisconsin,0400000US55,5914872,-555555555,55,2024
51,Wyoming,0400000US56,582397,-555555555,56,2024


In [9]:
DP05_df['pop_est_me'].value_counts()

pop_est_me
-555555555    780
0              52
Name: count, dtype: int64

Looks like there might be an issue with the way the margin of error is coming through... I'm not sure how much I'll actually be using those, so I'll leave it for now and maybe dig in to it later

In [10]:
DP05_df.to_csv('../Data/DP05.csv', index = False)

## DP03  

For easier manipulation and understanding, I'm going to create 2 tables out of DP03 - one with employment figures, and one with household income information

### Employment  
- DP03_0001E: Total (estimated) population over 16 years old
- DP03_0004E: Population over 16 in labor force & employed
- DP03_0005E: Population over 16 in labor force & unemployed
- DP03_0006E: Population over 16 in labor force in the armed forces
- DP03_0007E: Population over 16 not in the labor force

In [17]:
employ_params = {
    'get': 'NAME,GEO_ID,DP03_0001E,DP03_0001M,DP03_0004E,DP03_0004M,DP03_0005E,DP03_0005M,DP03_0006E,DP03_0006M,DP03_0007E,DP03_0007M', ## field names, separated by comma
    'for': 'state:*',
    'key': api_key
}

In [18]:
## Create empty dataframe to fill
employ_df = pd.DataFrame()

## loop through the years in range
for year in years:
    endpoint = 'https://api.census.gov/data/' + str(year) +'/acs/acs5/profile?'  ## create endpoint URL incorporating year
    response = requests.get(endpoint, params = employ_params).json()  ## get request from API
    df = pd.DataFrame(response) ## convert to dataframe
    df.columns = df.iloc[0] ## promote first row to headers
    df = df[1:] ## drop first row
    df['year'] = year ## add year

    employ_df = pd.concat([employ_df, df]) ## add current year dataframe to the overall dataframe

employ_df

,NAME,GEO_ID,DP03_0001E,DP03_0001M,DP03_0004E,DP03_0004M,DP03_0005E,DP03_0005M,DP03_0006E,DP03_0006M,DP03_0007E,DP03_0007M,state,year
1,Alabama,0400000US01,3639386,1272,2013755,6128,170118,3051,18206,1095,1437307,6046,01,2009
2,Alaska,0400000US02,521998,620,326950,2817,31342,1128,16640,1285,147066,2216,02,2009
3,Arizona,0400000US04,4830306,1707,2810448,7082,204107,3713,20291,991,1795460,6369,04,2009
4,Arkansas,0400000US05,2218097,1205,1249579,5069,99002,2462,7263,660,862253,4456,05,2009
5,California,0400000US06,27958467,3545,16550706,17889,1410973,9166,139269,3061,9857519,16417,06,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,Washington,0400000US53,6336814,2053,3819646,10278,204777,3755,64212,2290,2248179,8268,53,2024
49,West Virginia,0400000US54,1467075,912,744248,5062,42641,1755,2884,513,677302,4931,54,2024
50,Wisconsin,0400000US55,4805425,1414,3031042,7688,101728,2785,3830,435,1668825,7212,55,2024
51,Wyoming,0400000US56,466238,726,288713,2364,11113,877,3817,486,162595,2251,56,2024


In [19]:
employ_df = employ_df.rename(columns = {'DP03_0001E': 'pop_est_ov16', 'DP03_0001M': 'pop_est_ov16_me',
                                    'DP03_0004E': 'employed', 'DP03_0004M': 'employed_me',
                                    'DP03_0005E': 'unemployed', 'DP03_0005M': 'unemployed_me',
                                    'DP03_0006E': 'armed_forces', 'DP03_0006M': 'armed_forces_me',
                                    'DP03_0007E': 'non_labor_force', 'DP03_0007M': 'non_labor_force_me'})

In [20]:
employ_df

,NAME,GEO_ID,pop_est_ov16,pop_est_ov16_me,employed,employed_me,unemployed,unemployed_me,armed_forces,armed_forces_me,non_labor_force,non_labor_force_me,state,year
1,Alabama,0400000US01,3639386,1272,2013755,6128,170118,3051,18206,1095,1437307,6046,01,2009
2,Alaska,0400000US02,521998,620,326950,2817,31342,1128,16640,1285,147066,2216,02,2009
3,Arizona,0400000US04,4830306,1707,2810448,7082,204107,3713,20291,991,1795460,6369,04,2009
4,Arkansas,0400000US05,2218097,1205,1249579,5069,99002,2462,7263,660,862253,4456,05,2009
5,California,0400000US06,27958467,3545,16550706,17889,1410973,9166,139269,3061,9857519,16417,06,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,Washington,0400000US53,6336814,2053,3819646,10278,204777,3755,64212,2290,2248179,8268,53,2024
49,West Virginia,0400000US54,1467075,912,744248,5062,42641,1755,2884,513,677302,4931,54,2024
50,Wisconsin,0400000US55,4805425,1414,3031042,7688,101728,2785,3830,435,1668825,7212,55,2024
51,Wyoming,0400000US56,466238,726,288713,2364,11113,877,3817,486,162595,2251,56,2024


In [21]:
employ_df.to_csv('../Data/DP03_employ.csv')

### Household Income  
- DP03_0051E: Total (est) number of households
- DP03_0052E: # of households with income... < &#36;10k  
- DP03_0053E: ... between &#36;10k - 14.9k
- DP03_0054E: ... between &#36;15k - 24.9k
- DP03_0055E: ... between &#36;25k - 34.9k
- DP04_0056E: ... between &#36;35k - 49.9k
- DP03_0057E: ... between &#36;50 - 74.9k
- DP03_0058E: ... between &#36;75k - 99.9k
- DP03_0059E: ... between &#36;100k - 149.9k
- DP03_0060E: ... between &#36;150k - 199.9k
- DP03_0061E: ... over >= &#36;200k
- DP03_0062E: median household income
- DP03_0063E: mean household income

In [22]:
income_params = {
    'get': 'NAME,GEO_ID,DP03_0051E,DP03_0051M,DP03_0052E,DP03_0052M,DP03_0053E,DP03_0053M,DP03_0054E,DP03_0054M,DP03_0055E,DP03_0055M,DP03_0056E,DP03_0056M,DP03_0057E,DP03_0057M,DP03_0058E,DP03_0058M,DP03_0059E,DP03_0059M,DP03_0060E,DP03_0060M,DP03_0061E,DP03_0061M,DP03_0062E,DP03_0062M,DP03_0063E,DP03_0063M',
    'for': 'state:*',
    'key': api_key
}

In [31]:
## the variable codes changed in 2010, so I'm going to define separate parameters for getting the 2009 data
income_params_2009 = {
    'get': 'NAME,GEO_ID,DP03_0052E,DP03_0052M,DP03_0053E,DP03_0053M,DP03_0054E,DP03_0054M,DP03_0055E,DP03_0055M,DP03_0056E,DP03_0056M,DP03_0057E,DP03_0057M,DP03_0058E,DP03_0058M,DP03_0059E,DP03_0059M,DP03_0060E,DP03_0060M,DP03_0061E,DP03_0061M,DP03_0062E,DP03_0062M,DP03_0063E,DP03_0063M,DP03_0064E,DP03_0064M',
    'for': 'state:*',
    'key': api_key
}

In [48]:
## define names based on variable codes
column_names = {'2009': {'DP03_0052E': 'est_households', 'DP03_0052M': 'est_households_me' ,
                        'DP03_0053E': 'under_10k', 'DP03_0053M': 'under_10k_me',
                        'DP03_0054E': 'bw_10_15k', 'DP03_0054M': 'bw_10_15k_me',
                        'DP03_0055E': 'bw_15_25k', 'DP03_0055M': 'bw_15_25k_me',
                        'DP03_0056E': 'bw_25_35k', 'DP03_0056M': 'bw_25_35k_me',
                        'DP03_0057E': 'bw_35_50k', 'DP03_0057M': 'bw_35_50k_me',
                        'DP03_0058E': 'bw_50_75k', 'DP03_0058M': 'bw_50_75k_me',
                        'DP03_0059E': 'bw_75_100k', 'DP03_0059M': 'bw_75_100k_me',
                        'DP03_0060E': 'bw_100_150k', 'DP03_0060M': 'bw_100_150k_me',
                        'DP03_0061E': 'bw_150_200k', 'DP03_0061M': 'bw_150_200k_me',
                        'DP03_0062E': 'over_200k', 'DP03_0062M': 'over_200k_me',
                        'DP03_0063E': 'median_hh_income', 'DP03_0063M': 'median_hh_income_me',
                        'DP03_0064E': 'mean_hh_income', 'DP03_0064M': 'mean_hh_income_me'},
                '2010': {'DP03_0051E': 'est_households', 'DP03_0051M': 'est_households_me' ,
                        'DP03_0052E': 'under_10k', 'DP03_0052M': 'under_10k_me',
                        'DP03_0053E': 'bw_10_15k', 'DP03_0053M': 'bw_10_15k_me',
                        'DP03_0054E': 'bw_15_25k', 'DP03_0054M': 'bw_15_25k_me',
                        'DP03_0055E': 'bw_25_35k', 'DP03_0055M': 'bw_25_35k_me',
                        'DP03_0056E': 'bw_35_50k', 'DP03_0056M': 'bw_35_50k_me',
                        'DP03_0057E': 'bw_50_75k', 'DP03_0057M': 'bw_50_75k_me',
                        'DP03_0058E': 'bw_75_100k', 'DP03_0058M': 'bw_75_100k_me',
                        'DP03_0059E': 'bw_100_150k', 'DP03_0059M': 'bw_100_150k_me',
                        'DP03_0060E': 'bw_150_200k', 'DP03_0060M': 'bw_150_200k_me',
                        'DP03_0061E': 'over_200k', 'DP03_0061M': 'over_200k_me',
                        'DP03_0062E': 'median_hh_income', 'DP03_0062M': 'median_hh_income_me',
                        'DP03_0063E': 'mean_hh_income', 'DP03_0063M': 'mean_hh_income_me'}}

In [50]:
## Create empty dataframe to fill
income_df = pd.DataFrame()

## loop through the years in range
for year in years:
    endpoint = 'https://api.census.gov/data/' + str(year) +'/acs/acs5/profile?'  ## create endpoint URL incorporating year
    
    if year == 2009: ## request data from API, using appropriate parameters based on year
        response = requests.get(endpoint, params = income_params_2009).json()
    else:
        response = requests.get(endpoint, params = income_params).json()  
    
    df = pd.DataFrame(response) ## convert to dataframe
    df.columns = df.iloc[0] ## promote first row to headers
    df = df[1:] ## drop first row
    
    if year == 2009: ## rename columns based on variable labels for that year
        df = df.rename(columns = column_names.get('2009'))
    else:
        df = df.rename(columns = column_names.get('2010'))
    
    df['year'] = year ## add year column

    income_df = pd.concat([income_df, df]) ## add current year dataframe to the overall dataframe

income_df

,NAME,GEO_ID,est_households,est_households_me,under_10k,under_10k_me,bw_10_15k,bw_10_15k_me,bw_15_25k,bw_15_25k_me,...,bw_150_200k,bw_150_200k_me,over_200k,over_200k_me,median_hh_income,median_hh_income_me,mean_hh_income,mean_hh_income_me,state,year
1,Alabama,0400000US01,1819441,5562,193469,3490,134988,2286,245224,3045,...,46035,1250,39992,1115,41216,231,56458,298,01,2009
2,Alaska,0400000US02,234779,1453,9983,622,9008,623,18258,828,...,14169,598,9559,592,64635,747,79908,1128,02,2009
3,Arizona,0400000US04,2248170,6020,152636,2947,113771,2625,246562,4058,...,81181,1682,76876,1872,50296,228,67331,312,04,2009
4,Arkansas,0400000US05,1109635,4219,115369,2155,87121,1863,161883,2823,...,22426,968,19484,877,38542,288,52198,310,05,2009
5,California,0400000US06,12187191,20589,648213,6244,628340,5246,1154118,8336,...,754969,6369,767989,5646,60392,154,82948,200,06,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,Washington,0400000US53,3064820,5874,120544,3157,77938,2373,141938,2931,...,348680,4640,567560,5632,98141,414,134588,663,53,2024
49,West Virginia,0400000US54,728655,3249,45212,1538,40377,1601,68485,1974,...,45935,1509,42898,1656,59608,691,81190,817,54,2024
50,Wisconsin,0400000US55,2479480,5787,102887,2764,75277,1944,146516,2665,...,232818,3243,228753,3526,77485,455,101466,519,55,2024
51,Wyoming,0400000US56,243318,1490,11032,846,7422,647,16422,1144,...,23223,1128,20782,1139,76176,1168,100641,2126,56,2024


In [51]:
income_df.to_csv('../Data/DP03_income.csv')

## National Summaries  
I'm also going to pull the national values for:
- median household income
- mean household income
- median home value (owner-occupied units)
- median monthly owner cost (owner-occupied units with mortgage)
- median monthly owner cost (owner-occupied units without mortgage)
- median gross rent

Across these variables there are two times codes change - in 2010 and 2015 - so I'll need three different sets of parameters depending on which ones I need to get.

In [60]:
national_params_2009 = {
    'get': 'NAME,GEO_ID,DP03_0063E,DP03_0063M,DP03_0064E,DP03_0064M,DP04_0088E,DP04_0088M,DP04_0100E,DP04_0100M,DP04_0107E,DP04_0107M,DP04_0132E,DP04_0132M',
    'for': 'us:*',
    'key': api_key
}

In [61]:
national_params_2010 = {
    'get': 'NAME,GEO_ID,DP03_0062E,DP03_0062M,DP03_0063E,DP03_0063M,DP04_0088E,DP04_0088M,DP04_0100E,DP04_0100M,DP04_0107E,DP04_0107M,DP04_0132E,DP04_0132M',
    'for': 'us:*',
    'key': api_key
}

In [73]:
national_params = {
    'get': 'NAME,GEO_ID,DP03_0062E,DP03_0062M,DP03_0063E,DP03_0063M,DP04_0089E,DP04_0089M,DP04_0101E,DP04_0101M,DP04_0109E,DP04_0109M,DP04_0134E,DP04_0134M',
    'for': 'us:*',
    'key': api_key
}

In [74]:
## define names based on variable codes
column_names = {'2009': {'DP03_0063E': 'med_hh_income', 'DP03_0063M': 'med_hh_income_me',
                         'DP03_0064E': 'mean_hh_income', 'DP03_0064M': 'mean_hh_income_me',
                         'DP04_0088E': 'med_home_value', 'DP04_0088M': 'med_home_value_me',
                         'DP04_0100E': 'med_SMOC_mort', 'DP04_0100M': 'med_SMOC_mort_me',
                         'DP04_0107E': 'med_SMOC_no_mort', 'DP04_0107M': 'med_SMOC_no_mort_me',
                         'DP04_0132E': 'med_rent', 'DP04_0132M': 'med_rent_me'},
                '2010': {'DP03_0062E': 'med_hh_income', 'DP03_0062M': 'med_hh_income_me',
                         'DP03_0063E': 'mean_hh_income', 'DP03_0063M': 'mean_hh_income_me',
                         'DP04_0088E': 'med_home_value', 'DP04_0088M': 'med_home_value_me',
                         'DP04_0100E': 'med_SMOC_mort', 'DP04_0100M': 'med_SMOC_mort_me',
                         'DP04_0107E': 'med_SMOC_no_mort', 'DP04_0107M': 'med_SMOC_no_mort_me',
                         'DP04_0132E': 'med_rent', 'DP04_0132M': 'med_rent_me'},
                '2015': {'DP03_0062E': 'med_hh_income', 'DP03_0062M': 'med_hh_income_me',
                         'DP03_0063E': 'mean_hh_income', 'DP03_0063M': 'mean_hh_income_me',
                         'DP04_0089E': 'med_home_value', 'DP04_0089M': 'med_home_value_me',
                         'DP04_0101E': 'med_SMOC_mort', 'DP04_0101M': 'med_SMOC_mort_me',
                         'DP04_0109E': 'med_SMOC_no_mort', 'DP04_0109M': 'med_SMOC_no_mort_me',
                         'DP04_0134E': 'med_rent', 'DP04_0134M': 'med_rent_me'}}

In [76]:
## Create empty dataframe to fill
national_df = pd.DataFrame()

## loop through the years in range
for year in years:
    endpoint = 'https://api.census.gov/data/' + str(year) +'/acs/acs5/profile?'  ## create endpoint URL incorporating year

    if year == 2009:  ## request data from API, changing variables based on year
        response = requests.get(endpoint, params = national_params_2009).json()
    elif (year > 2009) and (year < 2015):
        response = requests.get(endpoint, params = national_params_2010).json()
    else:
        response = requests.get(endpoint, params = national_params).json()

    
    df = pd.DataFrame(response) ## convert to dataframe
    df.columns = df.iloc[0] ## promote first row to headers
    df = df[1:] ## drop first row

    if year == 2009: ## rename columns based on variable labels for that year
        df = df.rename(columns = column_names.get('2009'))
    elif (year > 2009) and (year < 2015):
        df = df.rename(columns = column_names.get('2010'))
    else:
        df = df.rename(columns = column_names.get('2015'))
    
    df['year'] = year ## add year

    national_df = pd.concat([national_df, df]) ## add current year dataframe to the overall dataframe

national_df

,NAME,GEO_ID,med_hh_income,med_hh_income_me,mean_hh_income,mean_hh_income_me,med_home_value,med_home_value_me,med_SMOC_mort,med_SMOC_mort_me,med_SMOC_no_mort,med_SMOC_no_mort_me,med_rent,med_rent_me,us,year
1,United States,0100000US,51425,83,70096,105,185400,201,1486,1,419,1,817,1,1,2009
1,United States,0100000US,51914,89,70883,123,188400,184,1524,1,431,1,841,1,1,2010
1,United States,0100000US,52762,99,72555,125,186200,156,1560,2,444,1,871,3,1,2011
1,United States,0100000US,53046,85,73034,122,181400,169,1559,2,449,2,889,1,1,2012
1,United States,0100000US,53046,89,73487,125,176700,198,1540,1,452,1,904,1,1,2013
1,United States,0100000US,53482,95,74596,124,175700,224,1522,2,457,1,920,1,1,2014
1,United States,0100000US,53889,110,75558,134,178600,133,1492,1,458,1,928,1,1,2015
1,United States,0100000US,55322,120,77866,146,184700,160,1491,1,462,1,949,1,1,2016
1,United States,0100000US,57652,138,81283,159,193500,156,1515,1,474,1,982,1,1,2017
1,United States,0100000US,60293,140,84938,164,204900,173,1558,1,490,1,1023,2,1,2018


In [77]:
national_df.to_csv('../Data/national_income_housing_summaries.csv')